# TF-IDF and Cosine Similarity — Completed Activity

**Source:** [mateenyaqoob/unstructuredata_tfidf_repo](https://github.com/mateenyaqoob/unstructuredata_tfidf_repo), commit `7dbd3dce293ab10fa3d61201999a9f43e020db05`.

This notebook completes **all six tasks** in `task_to_do.txt`, including the optional challenge, and answers the activity script's exit questions. The original ten documents are embedded unchanged, so this file also runs independently of the repository. All output cells are saved after execution.

**Run:** Open in Jupyter, VS Code, or Google Colab and choose **Run All**. The first code cell checks the active Python kernel and automatically installs any missing or outdated dependencies into that same environment before the imports run. The first run needs internet access if packages must be installed; subsequent runs use the installed packages. The dataset is embedded, so no data files or API keys are needed. Use a Python 3.9 or newer kernel.

| Task | Completed work |
|---|---|
| 1 | Load ten documents; check titles and word counts |
| 2 | Build and explain the English stop-word TF-IDF matrix |
| 3 | Display a labeled matrix sample and five weighted terms per document |
| 4 | Rank all documents for each of the three required queries |
| 5 | Compare the required bigram/filter variant with the baseline |
| 6 | Run the repository's three unchanged pytest tests and notebook checks |


## Automatic environment setup — run this first

This cell uses only Python's standard library. It checks NumPy, pandas, scikit-learn, pytest, and IPython before importing them. When installation is needed, `sys.executable -m pip` targets the **same Python interpreter as this notebook's kernel**, avoiding the common problem of installing into a different environment. Already compatible installations are reused. An installation failure stops here with a setup-specific explanation instead of failing later on a missing import.


In [1]:
import sys
import importlib
import importlib.util
import importlib.metadata
import re
import subprocess

if sys.version_info < (3, 9):
    raise RuntimeError("Select a Python 3.9 or newer notebook kernel, then run all cells again.")

DEPENDENCIES = [
    ("numpy", "numpy", "1.24.0"),
    ("pandas", "pandas", "2.0.0"),
    ("scikit-learn", "sklearn", "1.3.0"),
    ("pytest", "pytest", "7.4.0"),
    ("ipython", "IPython", "8.0.0"),
]

def release_tuple(version):
    parts = [int(part) for part in re.match(r"[0-9]+(?:\.[0-9]+)*", version).group().split(".")]
    return tuple((parts + [0, 0, 0])[:3])

needs_install = []
for package, module, minimum in DEPENDENCIES:
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if (importlib.util.find_spec(module) is None or installed is None
            or release_tuple(installed) < release_tuple(minimum)):
        needs_install.append(f"{package}>={minimum}")

if not needs_install:
    import_check = subprocess.run(
        [sys.executable, "-c", "import numpy, pandas, sklearn, pytest; from IPython.display import display, Markdown"],
        capture_output=True, text=True,
    )
    if import_check.returncode:
        needs_install.append("repair incompatible package installation")

if needs_install:
    print("Installing dependencies into the active notebook kernel:")
    print(sys.executable)
    print(", ".join(needs_install), flush=True)
    if importlib.util.find_spec("pip") is None:
        ensurepip = subprocess.run(
            [sys.executable, "-m", "ensurepip", "--upgrade"],
            capture_output=True, text=True,
        )
        if ensurepip.returncode:
            raise RuntimeError(
                "This Python kernel has no pip and cannot bootstrap it. "
                "Use Google Colab or select a standard Python virtual environment "
                "with pip, then run all cells again.\n" + ensurepip.stderr
            )
    requirements = [f"{package}>={minimum}" for package, module, minimum in DEPENDENCIES]
    installation = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "--disable-pip-version-check", "--quiet",
         *requirements],
        capture_output=True, text=True,
    )
    if installation.returncode:
        raise RuntimeError(
            "Automatic dependency installation failed. The kernel needs internet "
            "access and permission to install packages. Use Google Colab or a writable "
            "Python virtual environment, then run all cells again.\n"
            + installation.stdout + installation.stderr
        )
    importlib.invalidate_caches()
    print("Dependencies installed successfully.")
else:
    print("All required dependencies are already installed.")

verification = subprocess.run(
    [sys.executable, "-c", "import numpy, pandas, sklearn, pytest; from IPython.display import display, Markdown"],
    capture_output=True, text=True,
)
if verification.returncode:
    raise RuntimeError("Package imports are still unavailable in this kernel. "
                       "Restart the kernel and run all cells again.\n" + verification.stderr)
print("Environment ready. Continue with the next cell, or use Run All.")


Installing dependencies into the active notebook kernel:
/Users/shahzad/Documents/Codex/2026-10-14/ca/work/fresh-notebook-env/bin/python
numpy>=1.24.0, pandas>=2.0.0, scikit-learn>=1.3.0, pytest>=7.4.0


Dependencies installed successfully.


Environment ready. Continue with the next cell, or use Run All.


In [2]:
import sys
import platform
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", "{:.6f}".format)
display(pd.DataFrame({
    "Package": ["Python", "NumPy", "pandas", "scikit-learn"],
    "Version": [platform.python_version(), np.__version__, pd.__version__, sklearn.__version__],
}))


,Package,Version
0,Python,3.12.14
1,NumPy,2.5.4
2,pandas,3.0.6
3,scikit-learn,1.9.1


## Task 1 — Load and validate the data

The following lists reproduce `data/documents.py`. A small loader uses the same interface as `src/load_data.py`. Word counts use whitespace splitting, matching the provided tests; the vectorizer's token counts can differ because it handles punctuation and stop words.


In [3]:
TITLES = ['AI in Healthcare',
 'Climate Change',
 'Basketball Strategy',
 'Italian Pasta',
 'Space Exploration',
 'Cybersecurity',
 'Music Theory',
 'Gardening Vegetables',
 'Economics Inflation',
 'Public Health']

DOCUMENTS = ['Artificial intelligence is transforming healthcare by helping doctors analyze medical images, '
 'predict disease risk, and personalize treatments. Machine learning models can scan thousands of '
 'patient records to find patterns that humans might miss. Hospitals use AI to reduce waiting '
 'times and improve diagnosis accuracy. However, AI systems require large, clean datasets and '
 'careful validation before clinical use. Privacy, bias, and accountability are major concerns. If '
 'used responsibly, AI can support clinicians rather than replace them, making healthcare faster, '
 'safer, and more equitable for diverse populations.',
 'Climate change is driven by greenhouse gas emissions from burning fossil fuels. Renewable energy '
 'sources such as solar, wind, and hydropower can reduce carbon pollution. Governments and '
 'businesses are investing in batteries, smart grids, and energy efficiency. Communities also need '
 'adaptation plans for heat waves, floods, and droughts. Climate policy involves trade-offs '
 'between economic growth and environmental protection. Individual actions matter, but large-scale '
 'systemic change is essential. Accurate data and international cooperation help track progress '
 'toward lowering emissions and limiting global temperature rise.',
 'Basketball strategy depends on spacing, passing, and player movement. Coaches design plays to '
 'create open shots and mismatches. Offenses use pick-and-rolls, screens, and fast breaks, while '
 'defenses switch, trap, or zone. Analytics now influences decisions about shot selection, lineup '
 'combinations, and pace. A team with strong chemistry often outperforms a group of individual '
 'stars. Rebounding and turnovers frequently decide close games. Effective communication on '
 'defense is as important as scoring ability. Successful teams practice repeatedly and adjust '
 'tactics based on opponent weaknesses and game situations.',
 'Italian pasta cooking begins with quality ingredients and simple techniques. A good sauce '
 'balances tomatoes, olive oil, garlic, herbs, and sometimes meat or cheese. Pasta should be '
 'cooked in salted water until al dente, then finished in the sauce for better flavor. Fresh basil '
 'and parmesan add aroma and richness. Different shapes suit different sauces: long pasta for '
 'light sauces, short tubes for chunky sauces. Regional traditions vary widely across Italy. Home '
 "cooks can improve by tasting, seasoning gradually, and respecting the pasta's texture rather "
 'than relying only on a timer.',
 'Space exploration has advanced through robotic missions and human spaceflight. Mars rovers study '
 'geology, atmosphere, and signs of ancient water. Telescopes observe distant galaxies, '
 'exoplanets, and black holes. Space agencies and private companies are developing reusable '
 'rockets to lower launch costs. Astronauts face radiation, isolation, and microgravity health '
 'risks. Future missions may establish lunar bases or send crews to Mars. International '
 'collaboration and data sharing accelerate discovery. Exploring space also produces technologies '
 'that benefit life on Earth, from satellite navigation to medical devices and materials science.',
 'Cybersecurity protects computers, networks, and personal data from attacks. Common threats '
 'include phishing emails, malware, ransomware, and weak passwords. Multi-factor authentication '
 'and regular software updates reduce risk. Organizations use firewalls, encryption, and '
 'monitoring systems to detect intrusions. Employees are often the weakest link, so security '
 'training is essential. Data breaches can damage finances and reputation. Strong cybersecurity '
 'requires layered defenses, incident response plans, and privacy laws. As attackers become more '
 'sophisticated, defenders must continuously learn new tools, share threat intelligence, and '
 'design systems that are secure by default.',
 'Music theory explains scales, chords, rhythm, and harmony. Musicians practice intervals, chord '
 'progressions, and ear training to perform and compose. Different genres use distinct rhythms and '
 'instrumentation. A melody becomes memorable through repetition, contrast, and tension. '
 'Improvisation requires both theoretical knowledge and listening skills. Recording software '
 'allows artists to edit, mix, and produce tracks at home. Live performance depends on timing, '
 'dynamics, and communication with bandmates. Learning music improves memory, coordination, and '
 'emotional expression. Consistent practice, feedback, and experimentation help musicians develop '
 'a personal style over time.',
 'Gardening vegetables requires healthy soil, sunlight, water, and patience. Compost adds '
 'nutrients and improves soil structure. Gardeners rotate crops to prevent pests and diseases. '
 'Tomatoes, peppers, beans, and lettuce grow well in many home gardens. Mulch conserves moisture '
 'and reduces weeds. Raised beds help control drainage and soil quality. Pests can be managed with '
 'beneficial insects, companion planting, and careful monitoring. Harvesting at the right time '
 'improves flavor and nutrition. A successful garden depends on observing local climate, planning '
 'seasons, and learning from each growing cycle year after year.',
 'Economics studies how people, firms, and governments allocate scarce resources. Inflation '
 'measures the rising cost of goods and services over time. Central banks adjust interest rates to '
 'influence borrowing, spending, and investment. High inflation reduces purchasing power, '
 'especially for low-income households. Unemployment and economic growth are key indicators of '
 'national health. Fiscal policy involves taxation and government spending. Global trade, supply '
 'chains, and energy prices affect local economies. Understanding incentives, markets, and data '
 'helps policymakers design stable policies. Economic decisions always involve trade-offs between '
 'efficiency, equity, and long-term sustainability.',
 'Public health focuses on preventing disease and improving community well-being. Sleep, exercise, '
 'and nutrition strongly influence physical and mental health. Regular activity strengthens the '
 'heart, muscles, and immune system. A balanced diet includes vegetables, fruits, whole grains, '
 'and protein. Chronic stress can harm sleep, mood, and metabolism. Vaccination, clean water, and '
 'health education reduce illness. Public health campaigns encourage healthy habits and early '
 'screening. Access to care and social support also shape outcomes. Small daily choices, '
 'supportive environments, and evidence-based policies can help populations live longer, '
 'healthier, and more productive lives.']

def load_documents():
    """Return copies of the original titles and document texts."""
    return list(TITLES), list(DOCUMENTS)

titles, documents = load_documents()
word_counts = pd.DataFrame({"Document": titles, "Word count": [len(d.split()) for d in documents]})
assert len(titles) == len(documents) == 10
assert len(set(titles)) == 10
assert (word_counts["Word count"] > 30).all()
print(f"Loaded {len(documents)} documents; all contain more than 30 words.")
display(word_counts)


Loaded 10 documents; all contain more than 30 words.


,Document,Word count
0,AI in Healthcare,85
1,Climate Change,83
2,Basketball Strategy,85
3,Italian Pasta,91
4,Space Exploration,86
5,Cybersecurity,88
6,Music Theory,87
7,Gardening Vegetables,89
8,Economics Inflation,89
9,Public Health,91


## Task 2 — Build the TF-IDF matrix

**Term frequency (TF)** measures how often a term occurs in one document. **Inverse document frequency (IDF)** gives more weight to terms that occur in fewer documents. Their product measures a term's importance within a document relative to this corpus.

Here, scikit-learn uses raw term counts, smoothed IDF, and then L2 normalization by default:

\[
\mathrm{idf}(t)=\log\left(\frac{1+N}{1+\mathrm{df}(t)}\right)+1,
\qquad w_{d,t}=\mathrm{count}(t,d)\,\mathrm{idf}(t),
\qquad x_{d,t}=\frac{w_{d,t}}{\sqrt{\sum_j w_{d,j}^2}}.
\]

Here, $N=10$ and $\mathrm{df}(t)$ is the number of documents containing the term. `stop_words="english"` removes common English words before feature construction. The default tokenizer lowercases text and keeps tokens with at least two word characters. It does not stem words, so `password` and `passwords` remain different tokens. Each row represents a document; each column represents a vocabulary term.


In [4]:
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)
terms = vectorizer.get_feature_names_out()

assert X.shape == (10, len(terms))
assert np.allclose(np.sqrt(X.multiply(X).sum(axis=1)).A1, 1.0)
print("Matrix shape:", X.shape)
print("Vocabulary size:", len(terms))
print("Nonzero weights:", X.nnz)
print(f"Zero entries: {100 * (1 - X.nnz / (X.shape[0] * X.shape[1])):.2f}%")


Matrix shape: (10, 537)
Vocabulary size: 537
Nonzero weights: 620
Zero entries: 88.45%


### Verify one weight by hand

The next cell checks the TF, document frequency, smoothed IDF, and normalized weight for `climate` in **Climate Change**. The denominator includes every retained term in that document, not just `climate`.


In [5]:
analyzer = vectorizer.build_analyzer()
tokenized_documents = [analyzer(d) for d in documents]
example_term = "climate"
doc_index = titles.index("Climate Change")
term_index = vectorizer.vocabulary_[example_term]
tf_count = tokenized_documents[doc_index].count(example_term)
document_frequency = sum(example_term in tokens for tokens in tokenized_documents)
idf_manual = np.log((1 + len(documents)) / (1 + document_frequency)) + 1
raw_weights = np.array([
    tokenized_documents[doc_index].count(term) * vectorizer.idf_[i]
    for i, term in enumerate(terms)
])
weight_manual = tf_count * idf_manual / np.linalg.norm(raw_weights)
assert np.isclose(idf_manual, vectorizer.idf_[term_index])
assert np.isclose(weight_manual, X[doc_index, term_index])
display(pd.DataFrame([{
    "Document": titles[doc_index], "Term": example_term, "TF count": tf_count,
    "Document frequency": document_frequency, "Smoothed IDF": idf_manual,
    "Manual TF-IDF": weight_manual, "Vectorizer TF-IDF": float(X[doc_index, term_index]),
}]))


,Document,Term,TF count,Document frequency,Smoothed IDF,Manual TF-IDF,Vectorizer TF-IDF
0,Climate Change,climate,2,2,2.299283,0.212175,0.212175


## Task 3 — Inspect the matrix and top terms

The full DataFrame has document titles as its index and vocabulary terms as columns. A zero means that a term does not occur in that document. Dense conversion is appropriate for this ten-document dataset; a large corpus should stay sparse.


In [6]:
df = pd.DataFrame(X.toarray(), columns=terms, index=titles)
df.index.name = "Document"
print("First three documents × first ten terms (alphabetical vocabulary order):")
display(df.iloc[:3, :10].round(6))

def top_terms_table(matrix, fitted_vectorizer, document_titles, n=5):
    """Return top positive weights, with alphabetical tie-breaking."""
    features = fitted_vectorizer.get_feature_names_out()
    records = []
    for i, title in enumerate(document_titles):
        weights = matrix.getrow(i).toarray().ravel()
        ordered = sorted(np.flatnonzero(weights), key=lambda j: (-weights[j], features[j]))[:n]
        for rank, j in enumerate(ordered, start=1):
            records.append({"Document": title, "Rank": rank,
                            "Term": features[j], "TF-IDF": float(weights[j])})
    return pd.DataFrame(records, columns=["Document", "Rank", "Term", "TF-IDF"])

top5 = top_terms_table(X, vectorizer, titles)
assert len(top5) == 50
display(top5.set_index(["Document", "Rank"]))


First three documents × first ten terms (alphabetical vocabulary order):


,ability,accelerate,access,accountability,accuracy,accurate,actions,activity,adaptation,add
Document,,,,,,,,,,
AI in Healthcare,0.000000,0.000000,0.000000,0.124600,0.124600,0.000000,0.000000,0.000000,0.000000,0.000000
Climate Change,0.000000,0.000000,0.000000,0.000000,0.000000,0.124795,0.124795,0.000000,0.124795,0.000000
Basketball Strategy,0.130137,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Term   TF-IDF
Document             Rank                         
AI in Healthcare     1                 ai 0.373801
                     2         healthcare 0.249201
                     3                use 0.164779
                     4     accountability 0.124600
                     5           accuracy 0.124600
Climate Change       1             change 0.249591
                     2          emissions 0.249591
                     3            climate 0.212175
                     4             energy 0.212175
                     5           accurate 0.124795
Basketball Strategy  1            ability 0.130137
                     2          analytics 0.130137
                     3         basketball 0.130137
                     4             breaks 0.130137
                     5          chemistry 0.130137
Italian Pasta        1              pasta 0.443095
                     2             sauces 0.332321
                     3              sauce 0.221547
                     4          different 0.188335
                     5                add 0.110774
Space Exploration    1              space 0.349043
                     2               mars 0.232695
                     3           missions 0.232695
                     4         accelerate 0.116348
                     5           advanced 0.116348
Cybersecurity        1      cybersecurity 0.249431
                     2            systems 0.212040
                     3               data 0.164931
                     4          attackers 0.124716
                     5            attacks 0.124716
Music Theory         1              music 0.246952
                     2          musicians 0.246952
                     3           practice 0.209932
                     4             allows 0.123476
                     5            artists 0.123476
Gardening Vegetables 1               soil 0.358051
                     2              pests 0.238701
                     3               year 0.238701
                     4           improves 0.202917
                     5               adds 0.119350
Economics Inflation  1          inflation 0.234536
                     2           spending 0.234536
                     3           economic 0.199377
                     4              trade 0.199377
                     5             affect 0.117268
Public Health        1             health 0.346216
                     2             public 0.232757
                     3              sleep 0.232757
                     4             access 0.116379
                     5           activity 0.116379

High-weight terms provide a compact lexical summary of each topic. Repeated topic words such as `pasta`, `space`, and `health` can receive substantial weight. Tied scores are common because many words occur once in only one document; alphabetical tie-breaking makes the display reproducible, but a tied term is not more important than another merely because it appears first.


## Task 4 — Query similarity challenge

A query is transformed with the **already fitted** vectorizer, preserving the vocabulary and corpus-derived IDF values. Cosine similarity compares the directions of the query and document vectors:

\[
\mathrm{cosine}(q,d)=\frac{q\cdot d}{\|q\|_2\|d\|_2}.
\]

With nonnegative TF-IDF weights the scores lie between 0 and 1. A larger score indicates more weighted term overlap; it is not a probability. Ties preserve the dataset's document order. A query containing no vocabulary terms yields zero scores and no evidence for selecting a document.


In [7]:
def rank_documents(query, fitted_vectorizer=None, matrix=None, document_titles=None):
    """Return all documents ranked by descending query cosine similarity."""
    if not isinstance(query, str):
        raise TypeError("query must be a string")
    fitted_vectorizer = vectorizer if fitted_vectorizer is None else fitted_vectorizer
    matrix = X if matrix is None else matrix
    document_titles = titles if document_titles is None else document_titles
    query_vector = fitted_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, matrix).ravel()
    order = np.argsort(-scores, kind="stable")
    return pd.DataFrame({
        "Rank": np.arange(1, len(document_titles) + 1),
        "Document": [document_titles[i] for i in order],
        "Cosine similarity": scores[order],
    })

queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet",
]
expected_winners = ["Climate Change", "Cybersecurity", "Public Health"]
baseline_results = {}
query_summaries = []
for query, expected in zip(queries, expected_winners):
    result = rank_documents(query)
    baseline_results[query] = result
    query_tokens = analyzer(query)
    matched = [t for t in query_tokens if t in vectorizer.vocabulary_]
    unknown = [t for t in query_tokens if t not in vectorizer.vocabulary_]
    display(Markdown(f"### Query: `{query}`"))
    print("In-vocabulary tokens:", matched)
    print("Out-of-vocabulary tokens:", unknown)
    display(result.set_index("Rank"))
    assert result.iloc[0]["Document"] == expected
    query_summaries.append({"Query": query, "Winner": result.iloc[0]["Document"],
                            "Score": result.iloc[0]["Cosine similarity"]})
print("All three expected first-place rankings verified.")
display(pd.DataFrame(query_summaries))


### Query: `renewable energy and climate policy`

In-vocabulary tokens: ['renewable', 'energy', 'climate', 'policy']
Out-of-vocabulary tokens: []


,Document,Cosine similarity
Rank,,
1,Climate Change,0.323458
2,Economics Inflation,0.095225
3,Gardening Vegetables,0.048458
4,AI in Healthcare,0.000000
5,Basketball Strategy,0.000000
6,Italian Pasta,0.000000
7,Space Exploration,0.000000
8,Cybersecurity,0.000000
9,Music Theory,0.000000


### Query: `password security and phishing`

In-vocabulary tokens: ['security', 'phishing']
Out-of-vocabulary tokens: ['password']


...

### Query: `healthy sleep exercise and diet`

In-vocabulary tokens: ['healthy', 'sleep', 'exercise', 'diet']
Out-of-vocabulary tokens: []


,Document,Cosine similarity
Rank,,
1,Public Health,0.284861
2,Gardening Vegetables,0.044702
3,AI in Healthcare,0.000000
4,Climate Change,0.000000
5,Basketball Strategy,0.000000
6,Italian Pasta,0.000000
7,Space Exploration,0.000000
8,Cybersecurity,0.000000
9,Music Theory,0.000000


All three expected first-place rankings verified.


,Query,Winner,Score
0,renewable energy and climate policy,Climate Change,0.323458
1,password security and phishing,Cybersecurity,0.176375
2,healthy sleep exercise and diet,Public Health,0.284861


**Interpretation of the three rankings**

1. **Climate Change** directly discusses renewable energy, climate, and policy. Economics can also overlap on `energy` and `policy`, but the climate document matches more of the query's topic vocabulary.
2. **Cybersecurity** contains `security` and `phishing`. The query token `password` is out of vocabulary because the original document says `passwords`. The other two terms are sufficient to identify the topic.
3. **Public Health** discusses `healthy`, `sleep`, `exercise`, and `diet`. Gardening also contains `healthy`, creating weaker lexical overlap.

These examples demonstrate retrieval by term overlap, including a limitation of exact-token matching.


In [8]:
unseen_query = "qzxvplm nrrzzxx"
unseen_result = rank_documents(unseen_query)
assert vectorizer.transform([unseen_query]).nnz == 0
assert (unseen_result["Cosine similarity"] == 0).all()
print(f"Query {unseen_query!r}: all ten scores are zero; no relevant winner can be inferred.")


Query 'qzxvplm nrrzzxx': all ten scores are zero; no relevant winner can be inferred.


## Task 5 — Optional challenge: bigrams and frequency filters

The required variant uses `ngram_range=(1, 2)`, `max_df=0.8`, and `min_df=2`:

- Unigrams and adjacent two-token phrases are candidate features.
- A feature must occur in at least two documents.
- A feature occurring in more than 80% of documents is discarded (more than eight of these ten).

We compare the baseline, a bigram-only extension without frequency filtering, and the exact required variant. The middle model helps distinguish the effect of adding phrases from the effect of removing rare features. Bigrams are generated after stop-word removal, so a two-token feature may span an original stop word.


In [9]:
vectorizer_bigrams = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_bigrams = vectorizer_bigrams.fit_transform(documents)
vectorizer_v2 = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), max_df=0.8, min_df=2,
)
X_v2 = vectorizer_v2.fit_transform(documents)

models = {
    "Baseline: unigrams": (vectorizer, X),
    "Unigrams + bigrams, no DF filtering": (vectorizer_bigrams, X_bigrams),
    "Required variant: bigrams + DF filtering": (vectorizer_v2, X_v2),
}
model_summary = []
for name, (model, matrix) in models.items():
    features = model.get_feature_names_out()
    model_summary.append({"Model": name, "Rows": matrix.shape[0],
                          "Vocabulary size": len(features),
                          "Bigrams": sum(" " in term for term in features),
                          "Nonzero weights": matrix.nnz})
display(pd.DataFrame(model_summary).set_index("Model"))

v2_terms = vectorizer_v2.get_feature_names_out()
v2_top5 = top_terms_table(X_v2, vectorizer_v2, titles)
print("Top five positive terms per document in the required variant:")
display(v2_top5.set_index(["Document", "Rank"]))

def compact_terms(top_table):
    return top_table.groupby("Document", sort=False)["Term"].agg(
        lambda items: ", ".join(items)
    ).reindex(titles)

print("Side-by-side top-term comparison:")
display(pd.DataFrame({"Baseline top terms": compact_terms(top5),
                      "Required variant top terms": compact_terms(v2_top5)}))

unfiltered_df = np.asarray((X_bigrams > 0).sum(axis=0)).ravel()
print("Candidate features removed by min_df=2:", int((unfiltered_df < 2).sum()))
print("Candidate features removed by max_df=0.8:", int((unfiltered_df > 8).sum()))
print("Retained bigrams:", [term for term in v2_terms if " " in term])

variant_query_summary = []
for query in queries:
    result = rank_documents(query, vectorizer_v2, X_v2, titles)
    query_features = vectorizer_v2.build_analyzer()(query)
    retained_features = [t for t in query_features if t in vectorizer_v2.vocabulary_]
    baseline = baseline_results[query]
    variant_query_summary.append({
        "Query": query, "Baseline winner": baseline.iloc[0]["Document"],
        "Baseline score": baseline.iloc[0]["Cosine similarity"],
        "Variant winner": (result.iloc[0]["Document"] if retained_features
                           else "No vocabulary match (all scores zero)"),
        "Variant score": result.iloc[0]["Cosine similarity"],
        "Retained query features": ", ".join(retained_features) or "None",
    })
display(pd.DataFrame(variant_query_summary))


,Rows,Vocabulary size,Bigrams,Nonzero weights
Model,,,,
Baseline: unigrams,10,537,0,620
"Unigrams + bigrams, no DF filtering",10,1182,645,1268
Required variant: bigrams + DF filtering,10,69,3,155


Top five positive terms per document in the required variant:


Term   TF-IDF
Document             Rank                          
AI in Healthcare     1                 use 0.391487
                     2             careful 0.251653
                     3               clean 0.251653
                     4             disease 0.251653
                     5             improve 0.251653
Climate Change       1             climate 0.386225
                     2              energy 0.386225
                     3            economic 0.193113
                     4     economic growth 0.193113
                     5          efficiency 0.193113
Basketball Strategy  1              adjust 0.299666
                     2               based 0.299666
                     3       communication 0.299666
                     4           decisions 0.299666
                     5            defenses 0.299666
Italian Pasta        1           different 0.621057
                     2              flavor 0.310528
                     3             improve 0.310528
                     4                long 0.310528
                     5             quality 0.310528
Space Exploration    1       international 0.501540
                     2             medical 0.501540
                     3              health 0.438789
                     4                data 0.390115
                     5               water 0.390115
Cybersecurity        1             systems 0.434773
                     2                data 0.338180
                     3            defenses 0.217386
                     4           essential 0.217386
                     5        intelligence 0.217386
Music Theory         1            practice 0.499421
                     2       communication 0.249710
                     3           different 0.249710
                     4            improves 0.249710
                     5                live 0.249710
Gardening Vegetables 1            improves 0.436050
                     2             careful 0.218025
                     3             climate 0.218025
                     4              flavor 0.218025
                     5             healthy 0.218025
Economics Inflation  1            economic 0.372024
                     2               trade 0.372024
                     3              adjust 0.186012
                     4           decisions 0.186012
                     5     economic growth 0.186012
Public Health        1              health 0.685500
                     2               based 0.195884
                     3               clean 0.195884
                     4             disease 0.195884
                     5             healthy 0.195884

Side-by-side top-term comparison:


,Baseline top terms,Required variant top terms
Document,,
AI in Healthcare,"ai, healthcare, use, accountability, accuracy","use, careful, clean, disease, improve"
Climate Change,"change, emissions, climate, energy, accurate","climate, energy, economic, economic growth, efficiency"
Basketball Strategy,"ability, analytics, basketball, breaks, chemistry","adjust, based, communication, decisions, defenses"
Italian Pasta,"pasta, sauces, sauce, different, add","different, flavor, improve, long, quality"
Space Exploration,"space, mars, missions, accelerate, advanced","international, medical, health, data, water"
Cybersecurity,"cybersecurity, systems, data, attackers, attacks","systems, data, defenses, essential, intelligence"
Music Theory,"music, musicians, practice, allows, artists","practice, communication, different, improves, live"
Gardening Vegetables,"soil, pests, year, improves, adds","improves, careful, climate, flavor, healthy"
Economics Inflation,"inflation, spending, economic, trade, affect","economic, trade, adjust, decisions, economic growth"


Candidate features removed by min_df=2: 1113
Candidate features removed by max_df=0.8: 0
Retained bigrams: ['economic growth', 'policy involves', 'trade offs']


,Query,Baseline winner,Baseline score,Variant winner,Variant score,Retained query features
0,renewable energy and climate policy,Climate Change,0.323458,Climate Change,0.557468,"energy, climate, policy"
1,password security and phishing,Cybersecurity,0.176375,No vocabulary match (all scores zero),0.000000,None
2,healthy sleep exercise and diet,Public Health,0.284861,Gardening Vegetables,0.218025,healthy


### What changed, and do bigrams help?

The baseline has **537** terms. Adding bigrams without frequency filters yields **1,182** features, including **645** bigrams. The exact required variant retains only **69** features, including three bigrams: `economic growth`, `policy involves`, and `trade offs`. Relative to the baseline, that is an **87.2%** reduction in vocabulary size. `min_df=2` removes 1,113 candidate features from the unfiltered unigram/bigram vocabulary; `max_df=0.8` removes none on this corpus.

Some surviving phrases provide more context than single words, particularly `economic growth`, but the top-term comparison becomes less descriptive overall. For example, `pasta` and `cybersecurity` disappear because they occur in only one document. Thus adding bigrams can improve interpretability, while the required frequency filtering discards many of this dataset's strongest topic cues.

The query comparison shows the practical effect:

- **Climate query:** Climate Change remains first; `renewable` is removed but `energy`, `climate`, and `policy` survive.
- **Security query:** no query features survive. Every document receives zero similarity; there is no evidence for a winner, regardless of the order in which the tied documents are listed.
- **Health query:** only `healthy` survives. Gardening Vegetables overtakes Public Health because `healthy` accounts for a larger normalized weight in Gardening's reduced document vector. `sleep`, `exercise`, and `diet` were removed.

The results below summarize the actual retained features and vocabulary reduction. Query scores from different vocabularies are not directly comparable as measures of accuracy: removing features also changes each vector's normalization. A higher variant score alone does not establish better retrieval. For these three required queries, the baseline gives the expected first-place topic in all three cases, while the required variant gives it in only one case.


In [10]:
retained_bigrams = [term for term in v2_terms if " " in term]
removed_count = len(terms) - len(v2_terms)
reduction_percent = 100 * removed_count / len(terms)
print(f"Baseline vocabulary: {len(terms)}; required variant: {len(v2_terms)}.")
print(f"Net reduction: {removed_count} features ({reduction_percent:.1f}%).")
if retained_bigrams:
    print("Surviving phrases:", ", ".join(retained_bigrams))
    print("These phrases provide some context, but rare topic words are still removed.")
else:
    print("No bigrams survive min_df=2 in this corpus.")
    print("Thus the required variant adds no phrase-level meaning to its retained vocabulary.")
print("Recommendation for this dataset: keep the baseline for the three topic queries;")
print("use the unfiltered bigram model to explore phrases, and evaluate on more queries")
print("before claiming a general retrieval improvement.")


Baseline vocabulary: 537; required variant: 69.
Net reduction: 468 features (87.2%).
Surviving phrases: economic growth, policy involves, trade offs
These phrases provide some context, but rare topic words are still removed.
Recommendation for this dataset: keep the baseline for the three topic queries;
use the unfiltered bigram model to explore phrases, and evaluate on more queries
before claiming a general retrieval improvement.


## Task 6 — Run the provided unit tests

This cell runs **the original three tests unchanged** using `python -m pytest -q`. For portability, exact snapshots of the repository's dataset, loader, helpers, and tests are written into an isolated temporary directory. The temporary copy is removed afterward. This allows the standalone notebook to run the original tests even when the cloned repository is not present.

The tests check ten sufficiently long documents, matrix/vocabulary agreement, and the climate query's first-place ranking. Additional checks below verify that the notebook's implementation agrees with the original helpers for all three queries.


In [11]:
REPOSITORY_SNAPSHOT = {'data/documents.py': '"""Dataset: 10 long-text documents for the TF-IDF class activity."""\n'
                      '\n'
                      'TITLES = [\n'
                      '    "AI in Healthcare",\n'
                      '    "Climate Change",\n'
                      '    "Basketball Strategy",\n'
                      '    "Italian Pasta",\n'
                      '    "Space Exploration",\n'
                      '    "Cybersecurity",\n'
                      '    "Music Theory",\n'
                      '    "Gardening Vegetables",\n'
                      '    "Economics Inflation",\n'
                      '    "Public Health",\n'
                      ']\n'
                      '\n'
                      'DOCUMENTS = [\n'
                      '    """Artificial intelligence is transforming healthcare by helping '
                      'doctors analyze medical images, predict disease risk, and personalize '
                      'treatments. Machine learning models can scan thousands of patient records '
                      'to find patterns that humans might miss. Hospitals use AI to reduce waiting '
                      'times and improve diagnosis accuracy. However, AI systems require large, '
                      'clean datasets and careful validation before clinical use. Privacy, bias, '
                      'and accountability are major concerns. If used responsibly, AI can support '
                      'clinicians rather than replace them, making healthcare faster, safer, and '
                      'more equitable for diverse populations.""",\n'
                      '\n'
                      '    """Climate change is driven by greenhouse gas emissions from burning '
                      'fossil fuels. Renewable energy sources such as solar, wind, and hydropower '
                      'can reduce carbon pollution. Governments and businesses are investing in '
                      'batteries, smart grids, and energy efficiency. Communities also need '
                      'adaptation plans for heat waves, floods, and droughts. Climate policy '
                      'involves trade-offs between economic growth and environmental protection. '
                      'Individual actions matter, but large-scale systemic change is essential. '
                      'Accurate data and international cooperation help track progress toward '
                      'lowering emissions and limiting global temperature rise.""",\n'
                      '\n'
                      '    """Basketball strategy depends on spacing, passing, and player '
                      'movement. Coaches design plays to create open shots and mismatches. '
                      'Offenses use pick-and-rolls, screens, and fast breaks, while defenses '
                      'switch, trap, or zone. Analytics now influences decisions about shot '
                      'selection, lineup combinations, and pace. A team with strong chemistry '
                      'often outperforms a group of individual stars. Rebounding and turnovers '
                      'frequently decide close games. Effective communication on defense is as '
                      'important as scoring ability. Successful teams practice repeatedly and '
                      'adjust tactics based on opponent weaknesses and game situations.""",\n'
                      '\n'
                      '    """Italian pasta cooking begins with quality ingredients and simple '
                      'techniques. A good sauce balances tomatoes, olive oil, garlic, herbs, and '
                      'sometimes meat or cheese. Pasta should be cooked in salted water until al '
                      'dente, then finished in the sauce for better flavor. Fresh basil and '
                      'parmesan add aroma and richness. Different shapes suit different sauces: '
                      'long pasta for light sauces, short tubes for chunky sauces. Regional '
                      'traditions vary widely across Italy. Home cooks can improve by tasting, '
                      "seasoning gradually, and respecting the pasta's texture rather than relying "
                      'only on a timer.""",\n'
                      '\n'
                      '    """Space exploration has advanced through robotic missions and human '
                      'spaceflight. Mars rovers study geology, atmosphere, and signs of ancient '
                      'water. Telescopes observe distant galaxies, exoplanets, and black holes. '
                      'Space agencies and private companies are developing reusable rockets to '
                      'lower launch costs. Astronauts face radiation, isolation, and microgravity '
                      'health risks. Future missions may establish lunar bases or send crews to '
                      'Mars. International collaboration and data sharing accelerate discovery. '
                      'Exploring space also produces technologies that benefit life on Earth, from '
                      'satellite navigation to medical devices and materials science.""",\n'
                      '\n'
                      '    """Cybersecurity protects computers, networks, and personal data from '
                      'attacks. Common threats include phishing emails, malware, ransomware, and '
                      'weak passwords. Multi-factor authentication and regular software updates '
                      'reduce risk. Organizations use firewalls, encryption, and monitoring '
                      'systems to detect intrusions. Employees are often the weakest link, so '
                      'security training is essential. Data breaches can damage finances and '
                      'reputation. Strong cybersecurity requires layered defenses, incident '
                      'response plans, and privacy laws. As attackers become more sophisticated, '
                      'defenders must continuously learn new tools, share threat intelligence, and '
                      'design systems that are secure by default.""",\n'
                      '\n'
                      '    """Music theory explains scales, chords, rhythm, and harmony. Musicians '
                      'practice intervals, chord progressions, and ear training to perform and '
                      'compose. Different genres use distinct rhythms and instrumentation. A '
                      'melody becomes memorable through repetition, contrast, and tension. '
                      'Improvisation requires both theoretical knowledge and listening skills. '
                      'Recording software allows artists to edit, mix, and produce tracks at home. '
                      'Live performance depends on timing, dynamics, and communication with '
                      'bandmates. Learning music improves memory, coordination, and emotional '
                      'expression. Consistent practice, feedback, and experimentation help '
                      'musicians develop a personal style over time.""",\n'
                      '\n'
                      '    """Gardening vegetables requires healthy soil, sunlight, water, and '
                      'patience. Compost adds nutrients and improves soil structure. Gardeners '
                      'rotate crops to prevent pests and diseases. Tomatoes, peppers, beans, and '
                      'lettuce grow well in many home gardens. Mulch conserves moisture and '
                      'reduces weeds. Raised beds help control drainage and soil quality. Pests '
                      'can be managed with beneficial insects, companion planting, and careful '
                      'monitoring. Harvesting at the right time improves flavor and nutrition. A '
                      'successful garden depends on observing local climate, planning seasons, and '
                      'learning from each growing cycle year after year.""",\n'
                      '\n'
                      '    """Economics studies how people, firms, and governments allocate scarce '
                      'resources. Inflation measures the rising cost of goods and services over '
                      'time. Central banks adjust interest rates to influence borrowing, spending, '
                      'and investment. High inflation reduces purchasing power, especially for '
                      'low-income households. Unemployment and economic growth are key indicators '
                      'of national health. Fiscal policy involves taxation and government '
                      'spending. Global trade, supply chains, and energy prices affect local '
                      'economies. Understanding incentives, markets, and data helps policymakers '
                      'design stable policies. Economic decisions always involve trade-offs '
                      'between efficiency, equity, and long-term sustainability.""",\n'
                      '\n'
                      '    """Public health focuses on preventing disease and improving community '
                      'well-being. Sleep, exercise, and nutrition strongly influence physical and '
                      'mental health. Regular activity strengthens the heart, muscles, and immune '
                      'system. A balanced diet includes vegetables, fruits, whole grains, and '
                      'protein. Chronic stress can harm sleep, mood, and metabolism. Vaccination, '
                      'clean water, and health education reduce illness. Public health campaigns '
                      'encourage healthy habits and early screening. Access to care and social '
                      'support also shape outcomes. Small daily choices, supportive environments, '
                      'and evidence-based policies can help populations live longer, healthier, '
                      'and more productive lives."""\n'
                      ']\n',
 'src/__init__.py': '"""TF-IDF class activity package."""\n',
 'src/load_data.py': '"""Utilities for loading the classroom dataset."""\n'
                     'from typing import List, Tuple\n'
                     '\n'
                     'from data.documents import TITLES, DOCUMENTS\n'
                     '\n'
                     '\n'
                     'def load_documents() -> Tuple[List[str], List[str]]:\n'
                     '    """Return (titles, documents)."""\n'
                     '    return TITLES, DOCUMENTS\n'
                     '\n'
                     '\n'
                     'if __name__ == "__main__":\n'
                     '    titles, docs = load_documents()\n'
                     '    print(f"Loaded {len(docs)} documents.")\n'
                     '    for t, d in zip(titles, docs):\n'
                     '        print(f"- {t}: {len(d.split())} words")\n',
 'src/tfidf_utils.py': '"""Reusable TF-IDF helpers for the class activity."""\n'
                       'from typing import List, Tuple\n'
                       '\n'
                       'import pandas as pd\n'
                       'from sklearn.feature_extraction.text import TfidfVectorizer\n'
                       'from sklearn.metrics.pairwise import cosine_similarity\n'
                       '\n'
                       '\n'
                       'def build_tfidf(\n'
                       '    documents: List[str],\n'
                       '    stop_words: str = "english",\n'
                       '    ngram_range: Tuple[int, int] = (1, 1),\n'
                       '    max_df: float = 1.0,\n'
                       '    min_df: int = 1,\n'
                       '):\n'
                       '    """Fit a TfidfVectorizer and return (vectorizer, matrix)."""\n'
                       '    vectorizer = TfidfVectorizer(\n'
                       '        stop_words=stop_words,\n'
                       '        ngram_range=ngram_range,\n'
                       '        max_df=max_df,\n'
                       '        min_df=min_df,\n'
                       '    )\n'
                       '    matrix = vectorizer.fit_transform(documents)\n'
                       '    return vectorizer, matrix\n'
                       '\n'
                       '\n'
                       'def to_dataframe(matrix, vectorizer, titles: List[str]) -> pd.DataFrame:\n'
                       '    """Convert a sparse TF-IDF matrix into a labeled DataFrame."""\n'
                       '    return pd.DataFrame(\n'
                       '        matrix.toarray(),\n'
                       '        columns=vectorizer.get_feature_names_out(),\n'
                       '        index=titles,\n'
                       '    )\n'
                       '\n'
                       '\n'
                       'def top_terms_per_document(df: pd.DataFrame, n: int = 5) -> dict:\n'
                       '    """Return {title: {term: score}} of the top-n TF-IDF terms per '
                       'document."""\n'
                       '    result = {}\n'
                       '    for title in df.index:\n'
                       '        top = df.loc[title].sort_values(ascending=False).head(n)\n'
                       '        result[title] = top.to_dict()\n'
                       '    return result\n'
                       '\n'
                       '\n'
                       'def rank_documents_for_query(\n'
                       '    query: str,\n'
                       '    vectorizer: TfidfVectorizer,\n'
                       '    matrix,\n'
                       '    titles: List[str],\n'
                       ') -> List[Tuple[str, float]]:\n'
                       '    """Return [(title, similarity_score)] sorted by cosine similarity '
                       '(desc)."""\n'
                       '    query_vec = vectorizer.transform([query])\n'
                       '    scores = cosine_similarity(query_vec, matrix).flatten()\n'
                       '    return sorted(zip(titles, scores), key=lambda x: x[1], reverse=True)\n',
 'tests/test_tfidf_utils.py': '"""Basic unit tests for the TF-IDF activity utilities."""\n'
                              'import sys\n'
                              'import os\n'
                              '\n'
                              'sys.path.insert(0, '
                              'os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\n'
                              '\n'
                              'from src.load_data import load_documents\n'
                              'from src.tfidf_utils import build_tfidf, rank_documents_for_query\n'
                              '\n'
                              '\n'
                              'def test_load_documents():\n'
                              '    titles, docs = load_documents()\n'
                              '    assert len(titles) == 10\n'
                              '    assert len(docs) == 10\n'
                              '    assert all(len(d.split()) > 30 for d in docs)  # "long text"\n'
                              '\n'
                              '\n'
                              'def test_tfidf_matrix_shape():\n'
                              '    _, docs = load_documents()\n'
                              '    vectorizer, matrix = build_tfidf(docs)\n'
                              '    assert matrix.shape[0] == 10\n'
                              '    assert matrix.shape[1] == '
                              'len(vectorizer.get_feature_names_out())\n'
                              '\n'
                              '\n'
                              'def test_query_ranking_matches_expected_topic():\n'
                              '    titles, docs = load_documents()\n'
                              '    vectorizer, matrix = build_tfidf(docs)\n'
                              '    ranking = rank_documents_for_query(\n'
                              '        "renewable energy and climate policy", vectorizer, matrix, '
                              'titles\n'
                              '    )\n'
                              '    assert ranking[0][0] == "Climate Change"\n'}

import tempfile
import subprocess
from pathlib import Path

with tempfile.TemporaryDirectory(prefix="tfidf_activity_tests_") as temporary_directory:
    test_root = Path(temporary_directory)
    for relative_path, source_text in REPOSITORY_SNAPSHOT.items():
        target = test_root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(source_text, encoding="utf-8")
    test_run = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"], cwd=test_root,
        capture_output=True, text=True, timeout=120,
    )
    print(test_run.stdout)
    if test_run.stderr:
        print(test_run.stderr)
    assert test_run.returncode == 0, "The original repository tests did not pass."


...                                                                      [100%]
3 passed in 0.51s



In [12]:
# Execute the unchanged repository helpers in an isolated namespace for comparison.
original_data = {}
exec(compile(REPOSITORY_SNAPSHOT["data/documents.py"], "data/documents.py", "exec"), original_data)
original_helpers = {}
exec(compile(REPOSITORY_SNAPSHOT["src/tfidf_utils.py"], "src/tfidf_utils.py", "exec"), original_helpers)
assert original_data["TITLES"] == titles
assert original_data["DOCUMENTS"] == documents
original_vectorizer, original_matrix = original_helpers["build_tfidf"](documents)
assert np.array_equal(original_vectorizer.get_feature_names_out(), terms)
assert np.allclose(original_matrix.toarray(), X.toarray())
assert np.isfinite(df.to_numpy()).all()
assert (df.to_numpy() >= 0).all()
assert np.all((np.asarray((X_v2 > 0).sum(axis=0)).ravel() >= 2)
              & (np.asarray((X_v2 > 0).sum(axis=0)).ravel() <= 8))
for query, expected in zip(queries, expected_winners):
    ranking = rank_documents(query)
    original_ranking = original_helpers["rank_documents_for_query"](
        query, original_vectorizer, original_matrix, titles,
    )
    assert len(ranking) == 10
    assert ranking["Document"].tolist() == [title for title, score in original_ranking]
    assert np.allclose(ranking["Cosine similarity"], [score for title, score in original_ranking])
    assert ranking.iloc[0]["Document"] == expected
    assert ranking["Cosine similarity"].is_monotonic_decreasing
    assert ranking["Cosine similarity"].between(0, 1 + 1e-12).all()
assert rank_documents("")["Cosine similarity"].eq(0).all()
print("Notebook checks passed: original data and helper agreement, matrix validity,")
print("required feature-frequency filters, all three rankings, and zero-vector queries.")


Notebook checks passed: original data and helper agreement, matrix validity,
required feature-frequency filters, all three rankings, and zero-vector queries.


## Exit-ticket answers and limitations

**1. What does a high TF-IDF score mean for a term in one document?**  
The term is relatively prominent in that document and relatively uncommon across the corpus, after normalizing the document vector. It is a useful lexical cue, not a guarantee that the term conveys the document's central meaning.

**2. Why does IDF reduce the weight of common words such as “the” or “data”?**  
A term appearing in many documents distinguishes topics poorly, so its IDF is lower. With this vectorizer, `the` is removed by the English stop-word list before IDF is calculated. `data` is retained and receives less IDF weight than a term found in only one document. Smoothed IDF is at least 1; it does not assign zero IDF to a term merely because it occurs everywhere.

**3. Name one real task where TF-IDF is useful.**  
Keyword-based document search: transform a user's query with a fitted corpus vectorizer and rank documents by cosine similarity. It is also useful for extracting keywords or providing features to a text classifier.

**Limitations demonstrated or implied here:**

- Unigrams ignore word order. Bigrams preserve a little local order, but do not capture general meaning.
- Synonyms do not automatically match, and inflections such as `password`/`passwords` differ without stemming or lemmatization.
- An unseen query token contributes no weight; a wholly unseen query has zero similarity to every document.
- These ten documents and three queries are classroom examples, not a general retrieval benchmark.
- Adding future documents requires refitting the vectorizer if the vocabulary and IDF are to reflect the updated corpus.

**Completion:** all six requested tasks, the optional comparison, and all exit-ticket questions are complete. The saved outputs record the results of a full sequential run.
